In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#### Load Dati Patologo

In [2]:
excel_path = "/home/ubuntu/giodir/digitalPathology/data/aiFlopp/Trento AIFLOPP.xlsx"
sheet_name = "CONFRONTO REFERTI DEFINITIVO"

In [3]:
# open excel sheet in pandas

df = pd.read_excel(excel_path, sheet_name=sheet_name, header=[0, 1])

/home/ubuntu/giodir/digitalPathology/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [4]:
df.head()

Unnamed: 0_level_0 REFERTI REGGIO                 \
          ID PATIENT    CODICE CASO REPERE/VETRINO   
0               TN01          Tn001              1   
1               TN01          Tn001              2   
2               TN01          Tn001              3   
3               TN01          Tn001              4   
4               TN01          Tn001              5   

                                                  \
  NUMERO FRUSTOLI PER REPERE/VETRINO     LETTORE   
0                                1.0  RAGAZZI M.   
1                                1.0  RAGAZZI M.   
2                                2.0  RAGAZZI M.   
3                                2.0  RAGAZZI M.   
4                                1.0  RAGAZZI M.   

                                                                                                                                                                                                                                                \
  DIAGNOSI (0=no tumore; 1=ASAP/ATYP che richiede approfondimento; 2=PIN di alto grado; 3=AK intraduttale; 4=positivo per adenocarcinoma acinare; 5=positvo per adenocarcinoma duttale; N.A.=frustolo senza evidenza di ghiandole prostatiche)   
0                                                  4                                                                                                                                                                                             
1                                                  4                                                                                                                                                                                             
2                                                  4                                                                                                                                                                                             
3                                                  4                                                                                                                                                                                             
4                                                  4                                                                                                                                                                                             

                                                                                                                                                                                                                                                            \
  DIAGNOSI DOPO IMMUNO (0=no tumore; 1=ASAP/ATYP che richiede approfondimento; 2=PIN di alto grado; 3=AK intraduttale; 4=positivo per adenocarcinoma acinare; 5=positvo per adenocarcinoma duttale; N.A.=frustolo senza evidenza di ghiandole prostatiche)   
0                                                NaN                                                                                                                                                                                                         
1                                                NaN                                                                                                                                                                                                         
2                                                NaN                                                                                                                                                                                                         
3                                                NaN                                                                                                                                                                                                         
4                                      

In [5]:
df.columns.to_list()

[('Unnamed: 0_level_0', 'ID PATIENT'),
 ('REFERTI REGGIO', 'CODICE CASO'),
 ('REFERTI REGGIO', 'REPERE/VETRINO'),
 ('REFERTI REGGIO', 'NUMERO FRUSTOLI PER REPERE/VETRINO'),
 ('REFERTI REGGIO', 'LETTORE'),
 ('REFERTI REGGIO',
  'DIAGNOSI (0=no tumore; 1=ASAP/ATYP che richiede approfondimento; 2=PIN di alto grado; 3=AK intraduttale; 4=positivo per adenocarcinoma acinare; 5=positvo per adenocarcinoma duttale; N.A.=frustolo senza evidenza di ghiandole prostatiche)'),
 ('REFERTI REGGIO',
  'DIAGNOSI DOPO IMMUNO (0=no tumore; 1=ASAP/ATYP che richiede approfondimento; 2=PIN di alto grado; 3=AK intraduttale; 4=positivo per adenocarcinoma acinare; 5=positvo per adenocarcinoma duttale; N.A.=frustolo senza evidenza di ghiandole prostatiche)'),
 ('REFERTI REGGIO', 'GLEASON Principale'),
 ('REFERTI REGGIO', 'GLEASON Secondario'),
 ('REFERTI REGGIO', '% PATTERN 3'),
 ('REFERTI REGGIO', '% PATTERN 4'),
 ('REFERTI REGGIO', '% PATTERN 5'),
 ('REFERTI REGGIO', 'LUNGHEZZA CORES BIOPTICI (mm)'),
 ('REFERT

In [6]:
columns_to_keep = [
    ('REFERTI REGGIO', 'CODICE CASO'),
    ('REFERTI REGGIO', 'REPERE/VETRINO'),
    ('REFERTI TRENTO', 'GG ISUP PER SINGOLO CORE'),
]

columns_new_names = {
    ('REFERTI REGGIO', 'CODICE CASO'): 'patient_id',
    ('REFERTI REGGIO', 'REPERE/VETRINO'): 'bersaglio',
    ('REFERTI TRENTO', 'GG ISUP PER SINGOLO CORE'): 'GG_trento',
}


normalized_cols = [tuple(x.strip() if isinstance(x, str) else x for x in col)
                   if isinstance(col, tuple) else col
                   for col in df.columns]
df.columns = pd.MultiIndex.from_tuples(normalized_cols)  # only if they are tuples

filtered_df = df[columns_to_keep]
filtered_df.columns = [columns_new_names.get(col, col) for col in filtered_df.columns]

In [7]:
filtered_df.head()

,patient_id,bersaglio,GG_trento
0,Tn001,1,5.0
1,Tn001,2,5.0
2,Tn001,3,5.0
3,Tn001,4,5.0
4,Tn001,5,5.0


In [8]:
# Assign -1 values to case where GG is missing (meaning there is no tumor)
filtered_df['GG_trento'] = filtered_df['GG_trento'].fillna(-1)

In [22]:
parsed_case_code = filtered_df["patient_id"].apply(lambda x: str(int(str(x)[2:])))
filtered_df['tn_bag_id'] = "TN_" + parsed_case_code + "_" + filtered_df["bersaglio"].astype(str)

In [23]:
filtered_df.head()

,patient_id,bersaglio,GG_trento,tn_bag_id
0,Tn001,1,5.0,TN_1_1
1,Tn001,2,5.0,TN_1_2
2,Tn001,3,5.0,TN_1_3
3,Tn001,4,5.0,TN_1_4
4,Tn001,5,5.0,TN_1_5


#### Load CAD Data

In [54]:
excel_path = "/home/ubuntu/giodir/digitalPathology/data/aiFlopp/AIFLOPP TABELLA TRENTO (4).xlsx"
sheet_name = "AIFORIA"

In [55]:
# open excel sheet in pandas

cad_df = pd.read_excel(excel_path, sheet_name=sheet_name, header=[0])

In [56]:
cad_df.head()

,Caso,ID PATIENT,REPERE/VETRINO,LOW CONFIDENCE: ADENOCARCINOMA (0=no; 1=sì) (Compilato solo in presenza di adenocarcinoma),DIAGNOSI (0=no tumore; 1=adenocarcinoma),GLEASON Principale,GLEASON Secondario,% PATTERN 3,% PATTERN 4,% PATTERN 5,LUNGHEZZA CORES BIOPTICI (mm),LUNGHEZZA TUMORE (mm),% TUMORE,INVASIONE PERINEURALE (0=assente; 1=presente),CRIBRIFORME (0=assente; 1=presente),GG ISUP PER SINGOLO REPERE/VETRINO,NOTE: per tutti i vetrini vengono misurati tutti i frustoli presenti
0,25-I-10859,TN01,1,0,1.0,5,4.0,0.07,39.5,0.07,23.0,21.9,95.4,-,-,5,NaN
1,25-I-10859,TN01,2,0,1.0,4,5.0,0.00,55.9,0.00,20.5,18.4,90.1,-,-,5,NaN
2,25-I-10859,TN01,3,0,1.0,4,5.0,0.70,88.1,0.70,26.7,26.0,97.4,-,-,5,NaN
3,25-I-10859,TN01,4,0,1.0,4,5.0,0.10,86.7,0.10,28.2,27.3,96.7,-,-,5,NaN
4,25-I-10859,TN01,5,0,1.0,4,5.0,0.20,82.9,0.20,12.9,11.8,91.1,-,-,5,NaN


In [57]:
cad_df.columns.to_list()

['Caso',
 'ID PATIENT',
 'REPERE/VETRINO',
 'LOW CONFIDENCE: ADENOCARCINOMA (0=no; 1=sì) (Compilato solo in presenza di adenocarcinoma)',
 'DIAGNOSI (0=no tumore; 1=adenocarcinoma)',
 'GLEASON Principale',
 'GLEASON Secondario',
 '% PATTERN 3',
 '% PATTERN 4',
 '% PATTERN 5',
 'LUNGHEZZA CORES BIOPTICI (mm)',
 'LUNGHEZZA TUMORE (mm)',
 '% TUMORE',
 'INVASIONE PERINEURALE (0=assente; 1=presente)',
 'CRIBRIFORME (0=assente; 1=presente)',
 'GG ISUP PER SINGOLO REPERE/VETRINO',
 'NOTE: per tutti i vetrini vengono misurati tutti i frustoli presenti']

In [58]:
columns_to_keep = [
    'ID PATIENT', 'REPERE/VETRINO', 'GG ISUP PER SINGOLO REPERE/VETRINO'
]

renamed_columns = {
    'ID PATIENT': 'patient_id',
    'REPERE/VETRINO': 'bersaglio',
    'GG ISUP PER SINGOLO REPERE/VETRINO': 'GG_aiforia'
}

cad_df = cad_df[columns_to_keep].rename(columns=renamed_columns)

In [59]:
cad_df.head()

,patient_id,bersaglio,GG_aiforia
0,TN01,1,5
1,TN01,2,5
2,TN01,3,5
3,TN01,4,5
4,TN01,5,5


In [60]:
parsed_case_code = cad_df["patient_id"].apply(lambda x: str(int(str(x)[2:])))
cad_df['tn_bag_id'] = "TN_" + parsed_case_code + "_" + cad_df["bersaglio"].astype(str)

In [61]:
cad_df["GG_aiforia"].value_counts(dropna=False).index.to_list()

['-', '5', '3', '2', '1', '4', 1, nan, 4, '- ']

In [62]:
cad_df["GG_aiforia"] = cad_df["GG_aiforia"].fillna("-")
cad_df.loc[cad_df["GG_aiforia"] == "-", "GG_aiforia"] = -1
cad_df.loc[cad_df["GG_aiforia"] == "- ", "GG_aiforia"] = -1
cad_df["GG_aiforia"] = cad_df["GG_aiforia"].astype(float)

In [63]:
cad_df

,patient_id,bersaglio,GG_aiforia,tn_bag_id
0,TN01,1,5.0,TN_1_1
1,TN01,2,5.0,TN_1_2
2,TN01,3,5.0,TN_1_3
3,TN01,4,5.0,TN_1_4
4,TN01,5,5.0,TN_1_5
...,...,...,...,...
461,TN40,8,-1.0,TN_40_8
462,TN40,9,-1.0,TN_40_9
463,TN40,10,-1.0,TN_40_10
464,TN40,11,-1.0,TN_40_11


#### Merge Datasets

In [64]:
# First, check if the tn_bag_id values are unique in both dataframes

print(cad_df['tn_bag_id'].is_unique)
print(filtered_df['tn_bag_id'].is_unique)

# Second, check the bag_ids are in both dataframes

print(cad_df['tn_bag_id'].isin(filtered_df['tn_bag_id']).all())
print(filtered_df['tn_bag_id'].isin(cad_df['tn_bag_id']).all())

print(len(cad_df))

True
True
True
True
466


In [65]:
merged_df = pd.merge(filtered_df[['tn_bag_id', 'GG_trento']], cad_df[['tn_bag_id', 'GG_aiforia']], on='tn_bag_id', how='outer', suffixes=('_trento', '_aiforia'))

In [66]:
print(len(merged_df))
merged_df.head()

466


,tn_bag_id,GG_trento,GG_aiforia
0,TN_10_1,-1.0,-1.0
1,TN_10_10,-1.0,-1.0
2,TN_10_11,-1.0,-1.0
3,TN_10_12,-1.0,1.0
4,TN_10_2,-1.0,-1.0


In [67]:
merged_df["difference"] = np.abs(merged_df['GG_trento'] - merged_df['GG_aiforia'])

In [68]:
merged_df["difference"].value_counts()

difference
0.0    345
1.0     53
2.0     51
5.0      7
3.0      6
4.0      3
6.0      1
Name: count, dtype: int64

In [72]:
features_dir = Path("/home/ubuntu/giodir/digitalPathology/data/features/uni_features_TN")

def find_bag_id(tn_bag_id):
    for file in features_dir.glob(f"{tn_bag_id}_*.npz"):
        return file.stem  # return the filename without extension
    return None  # if no file is found

merged_df['bag_id'] = merged_df['tn_bag_id'].apply(find_bag_id)

print("not matched")
print(merged_df[merged_df["bag_id"].isnull()])

not matched
Empty DataFrame
Columns: [tn_bag_id, GG_trento, GG_aiforia, difference, bag_id]
Index: []


In [73]:
# Filter to keep only the analyzed cases

features_dir = Path("/home/ubuntu/giodir/digitalPathology/data/features/uni_features_TN")

available_bags = {path.stem for path in features_dir.glob("*.npz")}
print("Analyzed bags:", len(available_bags))

avail_df = merged_df[merged_df['bag_id'].isin(available_bags)]

print(f"Total cases: {len(merged_df)}, Available cases: {len(avail_df)},")

Analyzed bags: 466
Total cases: 466, Available cases: 466,


## LABEL TYPE

In [74]:
## binary like 0 vs 1+
avail_df["binary_difference"] = (avail_df["difference"] > 0).astype(int)

# keep only the important differences (set 0 where the difference is 0, 1 if it is 2 or more and None if it is 1)
avail_df["important_difference"] = avail_df["difference"].apply(lambda x: 0 if x == 0 else (1 if x >= 2 else None))

In [75]:
avail_df["binary_difference"].value_counts()

binary_difference
0    345
1    121
Name: count, dtype: int64

In [76]:
# Save in csv the three files like (bag_id, label_col)

basedir = Path("/home/ubuntu/giodir/digitalPathology/data/labels/tn_cad_discordance_labels")
basedir.mkdir(exist_ok=True)


# binary diff
avail_df[["bag_id", "binary_difference"]].rename(columns={"binary_difference": "label"}).to_csv(
    basedir / "binary_diff_labels.csv", index=False)

# binary important diff
avail_df[["bag_id", "important_difference"]].rename(columns={"important_difference": "label"}).to_csv(
    basedir / "binary_important_diff_labels.csv", index=False)

# original diff
avail_df[["bag_id", "difference"]].rename(columns={"difference": "label"}).to_csv(
    basedir / "difference_labels.csv", index=False)
